In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
from pathlib import Path

import scipy
from scipy.ndimage import zoom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable

import flexiznam as flz

from cottage_analysis.analysis import spheres
from cottage_analysis.plotting import depth_selectivity_plots
from cottage_analysis.plotting.rsof_plots import plot_rsof_slice

In [ ]:
project_rev = "colasa_3d-vision_revisions"


flm_sess_rev = flz.get_flexilims_session(project_id=project_rev)

mice = flz.get_entities(datatype="mouse", flexilims_session=flm_sess_rev)
print(f"{len(mice)} mice in {project_rev}")

In [ ]:
EXAMPLE_SESSION = 'PZAG17.3a_S20250402'

In [ ]:
# Load ndf, trials_df_tm, trials_df_tm_no_cut and trials_df_tm_no_cut

from cottage_analysis.pipelines import pipeline_utils
from cottage_analysis.analysis import treadmill
from cottage_analysis.plotting.rsof_plots import plot_RS_OF_matrix, plot_RS_OF_fit

example_mouse, example_session = EXAMPLE_SESSION.split('_')

print(f"Loading data for example session: {example_session}, mouse: {example_mouse}")

# Load a version of trials_df_tm without cuttin the ramp
_, trials_df_tm_no_cut = treadmill.sync_all_recordings(
        session_name=f"{example_mouse}_{example_session}",
        project=project_rev,
        photodiode_protocol=5,
        filter_datasets={"anatomical_only": 3, "annotated": True},
        recording_type="two_photon", 
            cut_trial_end=None,
            trial_duration=None,
            acceleration_time=None
    )
_, trials_df_tm_cut = treadmill.sync_all_recordings(
        session_name=f"{example_mouse}_{example_session}",
        project=project_rev,
        photodiode_protocol=5,
        filter_datasets={"anatomical_only": 3, "annotated": True},
        recording_type="two_photon",
    )

# Find frame rate
suite2p_ds = flz.get_datasets(
    origin_name=f"{example_mouse}_{example_session}", 
    dataset_type='suite2p_rois',
    filter_datasets=dict(annotated=True),
    flexilims_session=flm_sess_rev,
    allow_multiple=False)
fs = suite2p_ds.extra_attributes['fs']


In [ ]:
[c for c in trials_df_tm_no_cut.columns if 'dff' in c]

In [ ]:
[c for c in trials_df_tm_no_cut.columns if 'OF' in c]

In [ ]:
# ---- Fit OF Decoder ----
from cottage_analysis.analysis import population_ridge_decoder

results_of = population_ridge_decoder.of_decoder(
    trials_df=trials_df_tm_no_cut,
    frame_rate=fs,
    closed_loop=1,
    rolling_window=None,  # No smoothing
    downsample_window=None,  # No downsampling
    log_transform=True,
    rs_thr=None,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
)

# Shuffle control
results_of_shuffle = population_ridge_decoder.of_decoder(
    trials_df=trials_df_tm_no_cut,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
    rs_thr=None,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
    shuffle_control=True,
)

# Add predictions back to trials_df
trials_df_tm_no_cut['OF_pred'] = results_of['y_pred_trials']
trials_df_tm_no_cut['OF_actual_log'] = results_of['y_test_trials']

print(f'\n=== OF Decoder ===')
print(f'R² = {results_of["r2"]:.3f}  (shuffle: {results_of_shuffle["r2"]:.3f})')
print(f'Pearson r = {results_of["pearson_r"]:.3f}  (shuffle: {results_of_shuffle["pearson_r"]:.3f})')


In [ ]:
# ---- Plot OF Decoder ----

# Concatenate all non-null predictions and actual values from the dataframe for plotting
actuals_of = np.concatenate(trials_df_tm_no_cut['OF_actual_log'].values)
predictions_of = np.concatenate(trials_df_tm_no_cut['OF_pred'].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lims = np.nanpercentile(actuals_of, [1,99])
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].scatter(actuals_of, predictions_of, alpha=0.1, s=1)
axes[0].set_xlabel('Actual log(OF) (rad/s)')
axes[0].set_ylabel('Predicted log(OF) (rad/s)')
axes[0].set_title(f'OF decoder: R² = {results_of["r2"]:.3f}, r = {results_of["pearson_r"]:.3f}')


# Plot a longer time series window (5000 frames)
n_show = min(5000, len(actuals_of))
axes[1].plot(actuals_of[:n_show], label='Actual', alpha=0.7)
axes[1].plot(predictions_of[:n_show], label='Predicted', alpha=0.7)
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('log(OF) (rad/s)')
axes[1].legend()
axes[1].set_title('Predicted vs Actual OF (time series)')

plt.tight_layout()
plt.show()


In [ ]:
# ---- Fit RS Decoder ----
from cottage_analysis.analysis import population_ridge_decoder

results_rs = population_ridge_decoder.rs_decoder(
    trials_df=trials_df_tm_no_cut,
    closed_loop=1,
    rolling_window=None,  # No smoothing
    downsample_window=None,  # No downsampling
    frame_rate=fs,
    log_transform=True,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
)

# Shuffle control
results_rs_shuffle = population_ridge_decoder.rs_decoder(
    trials_df=trials_df_tm_no_cut,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
    shuffle_control=True,
)

# Add predictions back to trials_df
trials_df_tm_no_cut['RS_pred'] = results_rs['y_pred_trials']
trials_df_tm_no_cut['RS_actual_log'] = results_rs['y_test_trials']

print(f'\n=== RS Decoder ===')
print(f'R² = {results_rs["r2"]:.3f}  (shuffle: {results_rs_shuffle["r2"]:.3f})')
print(f'Pearson r = {results_rs["pearson_r"]:.3f}  (shuffle: {results_rs_shuffle["pearson_r"]:.3f})')


In [ ]:
# ---- Plot RS Decoder ----

# Concatenate all non-null predictions and actual values from the dataframe for plotting
actuals_rs = np.concatenate(trials_df_tm_no_cut['RS_actual_log'].dropna().values)
predictions_rs = np.concatenate(trials_df_tm_no_cut['RS_pred'].dropna().values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lims = np.nanpercentile(actuals_rs, [1, 99])
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].scatter(actuals_rs, predictions_rs, alpha=0.1, s=1)
axes[0].set_xlabel('Actual log(RS) (m/s)')
axes[0].set_ylabel('Predicted log(RS) (m/s)')
axes[0].set_title(f'RS decoder: R² = {results_rs["r2"]:.3f}, r = {results_rs["pearson_r"]:.3f}')

# Plot a longer time series window (5000 frames)
n_show = min(5000, len(actuals_rs))
axes[1].plot(actuals_rs[:n_show], label='Actual', alpha=0.7)
axes[1].plot(predictions_rs[:n_show], label='Predicted', alpha=0.7)
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('log(RS) (m/s)')
axes[1].legend()
axes[1].set_title('Predicted vs Actual RS (time series)')
plt.tight_layout()
plt.show()


In [ ]:
# ---- Fit Depth Decoder ----
from cottage_analysis.analysis import population_ridge_decoder

results_depth = population_ridge_decoder.depth_decoder(
    trials_df=trials_df_tm_no_cut,
    closed_loop=1,
    rolling_window=None,  # No smoothing
    downsample_window=None,  # No downsampling
    frame_rate=fs,
    log_transform=True,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
)

# Shuffle control
results_depth_shuffle = population_ridge_decoder.depth_decoder(
    trials_df=trials_df_tm_no_cut,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
    alphas=[0.01, 0.1, 1, 10, 100, 1000],
    k_folds=5,
    random_state=42,
    shuffle_control=True,
)

# Add predictions back to trials_df
trials_df_tm_no_cut['depth_pred'] = results_depth['y_pred_trials']
trials_df_tm_no_cut['depth_actual_log'] = results_depth['y_test_trials']

print(f'\n=== Depth Decoder ===')
print(f'R² = {results_depth["r2"]:.3f}  (shuffle: {results_depth_shuffle["r2"]:.3f})')
print(f'Pearson r = {results_depth["pearson_r"]:.3f}  (shuffle: {results_depth_shuffle["pearson_r"]:.3f})')


In [ ]:
# ---- Plot Depth Decoder ----

# Concatenate all non-null predictions and actual values from the dataframe for plotting
actuals_depth = np.concatenate(trials_df_tm_no_cut['depth_actual_log'].dropna().values)
predictions_depth = np.concatenate(trials_df_tm_no_cut['depth_pred'].dropna().values)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lims = np.nanpercentile(actuals_depth, [1, 99])
axes[0].plot(lims, lims, 'r--', lw=1)
axes[0].scatter(actuals_depth, predictions_depth, alpha=0.1, s=1)
axes[0].set_xlabel('Actual log(depth) (m)')
axes[0].set_ylabel('Predicted log(depth) (m)')
axes[0].set_title(f'Depth decoder: R² = {results_depth["r2"]:.3f}, r = {results_depth["pearson_r"]:.3f}')

# Plot a longer time series window (5000 frames)
n_show = min(5000, len(actuals_depth))
axes[1].plot(actuals_depth[:n_show], label='Actual', alpha=0.7)
axes[1].plot(predictions_depth[:n_show], label='Predicted', alpha=0.7)
axes[1].set_xlabel('Frame')
axes[1].set_ylabel('log(depth) (m)')
axes[1].legend()
axes[1].set_title('Predicted vs Actual Depth (time series)')
plt.tight_layout()
plt.show()


In [ ]:
trials_df_tm_no_cut['expected_OF'] = trials_df_tm_no_cut.expected_optic_flow_stim.map(np.nanmedian)
trials_df_tm_no_cut['expected_RS'] = trials_df_tm_no_cut.MotorSpeed_stim.map(np.nanmedian)
trials_df_tm_no_cut['RS_error'] = trials_df_tm_no_cut.RS_stim.map(np.log) - trials_df_tm_no_cut.RS_pred
trials_df_tm_no_cut['OF_error'] = trials_df_tm_no_cut.OF_stim.map(np.log) - trials_df_tm_no_cut.OF_pred
trials_df_tm_no_cut['depth_error'] = trials_df_tm_no_cut.depth_actual_log - trials_df_tm_no_cut.depth_pred
trials_df_tm_no_cut['RS_error'] = (trials_df_tm_no_cut.RS_stim.map(np.log) - trials_df_tm_no_cut.RS_pred)**2
trials_df_tm_no_cut['OF_error'] = (trials_df_tm_no_cut.OF_stim.map(np.log) - trials_df_tm_no_cut.OF_pred)**2
trials_df_tm_no_cut['depth_error'] = (trials_df_tm_no_cut.depth_actual_log - trials_df_tm_no_cut.depth_pred)**2

In [ ]:
import warnings


trial_averaged_rows = []
for (of, rs), tdf in trials_df_tm_no_cut.groupby(['expected_OF', 'expected_RS']):
    part = tdf[['dff_stim', 'RS_stim', 'OF_stim', 'depth_actual_log', 'RS_pred', 'OF_pred', 'depth_pred', 'OF_error', 'RS_error', 'depth_error']]
    if len(part) == 0:
        continue
    
    averaged_row = {
        'expected_OF': of,
        'expected_RS': rs,
        'depth': tdf.depth.mean()
    }
    for col in part.columns:
        # Find minimum frame length in the group
        min_len = min(len(arr) for arr in part[col])
        # Truncate all trials to the minimum length, stack, and average
        truncated_arrays = [arr[:min_len] for arr in part[col]]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            averaged_row[col] = np.nanmean(np.stack(truncated_arrays), axis=0)
        
        
    trial_averaged_rows.append(averaged_row)
trial_averaged_df = pd.DataFrame(trial_averaged_rows)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import warnings

# Get sorted unique values for speeds and optic flows
unique_speeds = np.sort(trial_averaged_df['expected_RS'].unique())
unique_ofs = np.sort(trial_averaged_df['expected_OF'].unique())

n_rows = len(unique_speeds)
n_cols = len(unique_ofs)

# Create the grid of subplots
fig, axes = plt.subplots(
    nrows=n_rows, 
    ncols=n_cols, 
    figsize=(3.2 * n_cols, 2.5 * n_rows), 
    sharex=True, 
    sharey=True
)

# Ensure axes is always a 2D array
if n_rows == 1 and n_cols == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = np.expand_dims(axes, axis=0)
elif n_cols == 1:
    axes = np.expand_dims(axes, axis=1)

# Keep track of the first twin axis to share y-limits across all subplots
shared_ax_rs = None

# Plot in the corresponding subplots
for row_idx, speed in enumerate(unique_speeds):
    for col_idx, of in enumerate(unique_ofs):
        ax = axes[row_idx, col_idx]
        
        # Filter the trial_averaged_df for this combination
        subset = trial_averaged_df[
            (trial_averaged_df['expected_RS'] == speed) & 
            (trial_averaged_df['expected_OF'] == of)
        ]
        
        # Create twin axis for running speed
        ax_rs = ax.twinx()
        if shared_ax_rs is None:
            shared_ax_rs = ax_rs
            
        if len(subset) > 0:
            # 1. Plot median running speed on the twin axis (in background)
            rs_median = subset['RS_stim'].values[0]
            ax_rs.plot(rs_median, color='black', linestyle='--', alpha=0.25, label='Median RS')
            
            # 2. Plot the error traces on the main axis
            rs_err = subset['RS_error'].values[0]
            ax.plot(rs_err, color='#1f77b4', linewidth=2, label='RS error')
            
            of_err = subset['OF_error'].values[0]
            ax.plot(of_err, color='C1', linewidth=2, label='OF error')
            
            d_err = subset['depth_error'].values[0]
            ax.plot(d_err, color='C2', linewidth=2, label='Depth error')

        # Formatting twin axes ticks to avoid clutter:
        # Only show the y-ticks for the right-most column
        if col_idx < n_cols - 1:
            ax_rs.yaxis.set_ticklabels([])
        else:
            if row_idx == 0:
                ax_rs.set_ylabel('Speed (m/s)', fontsize=10)

        # Label the top row with expected OF columns
        if row_idx == 0:
            ax.set_title(f"expected OF: {of}", fontsize=10)
            
        # Label the right-most column with expected RS rows
        if col_idx == n_cols - 1:
            ax.text(
                1.25, 0.5, f"expected RS:\n{speed} cm/s", 
                transform=ax.transAxes, 
                va='center', ha='left', fontsize=10
            )
            
        ax.axhline(0, color='grey', zorder=-2, linestyle=':')

ax.legend(loc='upper right')
# Add global labels and adjust layout
fig.text(0.5, 0.01, 'Frames', ha='center', fontsize=12)
fig.text(0.01, 0.5, 'Decoder Error', va='center', rotation='vertical', fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
print(f'OF decoder: R² = {results_of["r2"]:.3f}, r = {results_of["pearson_r"]:.3f}')
print(f'RS decoder: R² = {results_rs["r2"]:.3f}, r = {results_rs["pearson_r"]:.3f}')
print(f'Depth decoder: R² = {results_depth["r2"]:.3f}, r = {results_depth["pearson_r"]:.3f}')
print(f'OF shuffle: R² = {results_of_shuffle["r2"]:.3f}')
print(f'RS shuffle: R² = {results_rs_shuffle["r2"]:.3f}')
print(f'Depth shuffle: R² = {results_depth_shuffle["r2"]:.3f}')


In [ ]:
# ---- Subsampling Analysis for all 3 targets ----
from cottage_analysis.analysis import population_ridge_decoder
subset_sizes=[5, 10, 20, 50, 75, 100, 200, 400, 600, "inf"]
n_resamples="auto"

# 1. Optic Flow (OF) decoder
print("Running subsampling for OF...")
sub_of = population_ridge_decoder.decode_with_neuron_subsets(
    trials_df=trials_df_tm_no_cut,
    decoder_func=population_ridge_decoder.of_decoder,
    subset_sizes=subset_sizes,
    n_resamples=n_resamples,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
    rs_thr=None,
)

# 2. Running Speed (RS) decoder
print("\nRunning subsampling for RS...")
sub_rs = population_ridge_decoder.decode_with_neuron_subsets(
    trials_df=trials_df_tm_no_cut,
    decoder_func=population_ridge_decoder.rs_decoder,
    subset_sizes=subset_sizes,
    n_resamples=n_resamples,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
)

# 3. Virtual Depth decoder (Sphere trials)
print("\nRunning subsampling for Depth...")
sub_depth = population_ridge_decoder.decode_with_neuron_subsets(
    trials_df=trials_df_tm_no_cut,
    decoder_func=population_ridge_decoder.depth_decoder,
    subset_sizes=subset_sizes,
    n_resamples=n_resamples,
    closed_loop=1,
    rolling_window=None,
    downsample_window=None,
    frame_rate=fs,
    log_transform=True,
)



In [ ]:
# ---- Plot Subsampling Curves Side-by-Side ----
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

targets = ["Optic Flow (OF)", "Running Speed (RS)", "Virtual Depth"]
results_list = [sub_of, sub_rs, sub_depth]
colors = ["tab:blue", "tab:orange", "tab:green"]

for i, (ax, res, title, color) in enumerate(zip(axes, results_list, targets, colors)):
    sizes = res["subset_sizes"]
    r2_means = res["r2_mean"]
    r2_stds = res["r2_std"]
    
    # Calculate SEM for subsets
    n_resamples = 5
    sems = [std / np.sqrt(n_resamples) for std in r2_stds[:-1]]
    
    # Plot resampled subsets with error bars (all sizes except the last one)
    ax.errorbar(sizes[:-1], r2_means[:-1], yerr=sems, fmt='o-', color=color,
                capsize=5, lw=2, elinewidth=1.5, label='Subsets (mean +/- SEM)')
    
    # Plot the full population decoding as a single distinct point with no error bar
    ax.plot(sizes[-1], r2_means[-1], 's', color='tab:red', markersize=8, label='Full Population')
    
    ax.axhline(0, color='gray', linestyle='--', alpha=0.7)
    ax.set_xlabel('Number of Neurons Included')
    if i == 0:
        ax.set_ylabel('Decoder R²')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

plt.suptitle('Decoder Performance vs. Neuron Population Size', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()
